# Notebook 07 — Overfitting Evidence and Training-Curve Analysis

**Reviewer-ready update:** this version keeps the previous loss/accuracy diagnostics and adds a Macro-F1 train-validation gap, because present-class Macro-F1 is the primary thesis metric. It also prefers the updated LSTM export when multiple LSTM output folders are available.


## Section 1 — Setup

This section imports the required libraries and defines the common constants used by the evaluation code. The same clients, feature list, class order, and sequence length are kept from the previous notebooks.


In [38]:
# 0.1 — Imports
import os, json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [39]:
# 0.2 — Constants
ACTIVE_CLIENTS = ["England", "Ireland", "USA"]
FEATURE_COLS   = ["NH3","BOD","DO","PO4","pH","Temp","Nitrogen","Nitrate"]
N_FEATS, N_CLASSES, WINDOW = 8, 5, 12
CLASS_NAMES = ["Excellent","Good","Fair","Marginal","Poor"]
BATCH_SIZE  = 512
OUTPUT_DIR  = Path("/kaggle/working/overfitting_evidence")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Section 2 — Input discovery

The notebook automatically searches the Kaggle input folders for the final preprocessing tensors and the final GRU, LSTM, and Transformer outputs. This avoids using old hardcoded paths after replacing earlier notebooks.


In [40]:

# 0.3 — Auto-discover paths with LSTM consistency safeguard
# This cell lists all candidate output folders and, when more than one LSTM output
# folder exists, prefers the newer/complete LSTM export that contains the added
# federated prediction files. NB07 itself uses local/centralized models and training
# histories, but this safeguard avoids accidentally reading an older LSTM export.

def find_dirs(marker):
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if marker in files:
            hits.append(Path(root))
    return sorted(hits)

def pick_first_or_fail(candidates, label):
    if not candidates:
        raise FileNotFoundError(f"{label} outputs not found.")
    if len(candidates) > 1:
        print(f"[INFO] Multiple {label} candidates found:")
        for c in candidates:
            print(f"  - {c}")
    return candidates[0]

def pick_lstm_dir(candidates):
    if not candidates:
        raise FileNotFoundError("N04 LSTM outputs not found.")
    complete = []
    for d in candidates:
        pred_dir = d.parent / "predictions"
        required = [
            pred_dir / f"lstm_fedprox_v1_{client.lower()}_test_preds.npz"
            for client in ACTIVE_CLIENTS
        ]
        if all(p.exists() for p in required):
            complete.append(d)
    if complete:
        if len(complete) > 1:
            print("[INFO] Multiple complete LSTM candidates found; using the first:")
            for c in complete:
                print(f"  - {c}")
        return sorted(complete)[0]
    if len(candidates) > 1:
        print("[WARN] Multiple LSTM candidates found, but none contains all FedProx_v1 prediction files.")
        print("       NB07 can still run because it uses training histories/models, but check the input source.")
        for c in candidates:
            print(f"  - {c}")
    return sorted(candidates)[0]

N02_DIR   = pick_first_or_fail(find_dirs("preprocess_manifest.json"), "N02")
GRU_DIR   = pick_first_or_fail(find_dirs("local_test_summary.csv"), "N03 GRU")
LSTM_DIR  = pick_lstm_dir(find_dirs("local_lstm_test_summary.csv"))
TX_DIR    = pick_first_or_fail(find_dirs("local_transformer_test_summary.csv"), "N05 Transformer")

TENSOR_DIR = N02_DIR / "tensors"
GRU_MODELS = GRU_DIR.parent / "models"
LSTM_MODELS = LSTM_DIR.parent / "models"
TX_MODELS   = TX_DIR.parent / "models"

print(f"N02 tensors : {TENSOR_DIR}")
print(f"GRU models  : {GRU_MODELS}")
print(f"LSTM models : {LSTM_MODELS}")
print(f"TX models   : {TX_MODELS}")


N02 tensors : /kaggle/input/datasets/hebacheblal/preprocess-outputs/tensors
GRU models  : /kaggle/input/datasets/hebacheblal/gru-baselines/models
LSTM models : /kaggle/input/datasets/hebacheblal/lstm-baselines/models
TX models   : /kaggle/input/datasets/hebacheblal/transformer-baselines/models


## Section 3 — Load training tensors

Only the training split is loaded here because the goal is to compare training behaviour against the validation metrics saved during training. The target labels are not modified, and no new preprocessing is applied.


In [41]:
# 1.1 — Load training tensors (train split only)
data = {}
for client in ACTIVE_CLIENTS:
    arr = np.load(TENSOR_DIR / f"{client.lower()}_stage1_sequences.npz")
    data[client] = {"X": arr["X_train"], "y": arr["y_train"]}
    arr.close()
    print(f"{client:<8}: train={data[client]['X'].shape[0]:,}")

England : train=76,530
Ireland : train=27,778
USA     : train=30,222


In [42]:
# 1.2 — Dataset + eval utility
class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def eval_model(model, X, y, weights=None):
    """Return loss, accuracy, and macro-F1 for a model on (X, y)."""
    loader = DataLoader(SeqDS(X, y), batch_size=BATCH_SIZE, shuffle=False)
    crit   = (nn.CrossEntropyLoss(weight=torch.tensor(weights, device=device))
              if weights is not None else nn.CrossEntropyLoss())
    model.eval()
    total_loss, total_n = 0.0, 0
    all_true, all_pred  = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits  = model(xb)
            loss    = crit(logits, yb)
            preds   = torch.argmax(logits, dim=1)
            total_loss += loss.item() * xb.size(0)
            total_n    += xb.size(0)
            all_true.append(yb.cpu().numpy())
            all_pred.append(preds.cpu().numpy())
    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    acc    = accuracy_score(y_true, y_pred)
    f1     = f1_score(y_true, y_pred, labels=list(range(N_CLASSES)),
                      average="macro", zero_division=0)
    return total_loss / total_n, acc, f1

## Section 4 — Model definitions

The model classes are redefined only so that the saved checkpoints can be loaded. Their layer names must match the models trained in N03, N04, and N05. No new model is trained in this notebook.


In [43]:
# 2 — Define all three model classes (names must match the saved state dicts exactly)
class GRUClassifier(nn.Module):
    """Matches the GRUClassifier saved in N03 models/: self.gru, self.dropout, self.classifier."""
    def __init__(self, input_size=N_FEATS, hidden_size=64,
                 num_layers=1, num_classes=N_CLASSES, dropout=0.2):
        super().__init__()
        self.gru        = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        _, h = self.gru(x)
        return self.classifier(self.dropout(h[-1]))


class LSTMClassifier(nn.Module):
    """Matches the LSTMClassifier saved in N04 models/: self.lstm, self.dropout, self.classifier."""
    def __init__(self, input_size=N_FEATS, hidden_size=64,
                 num_layers=1, num_classes=N_CLASSES, dropout=0.2):
        super().__init__()
        self.lstm       = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.classifier(self.dropout(h[-1]))


class PositionalEncoding(nn.Module):
    """Matches PositionalEncoding saved inside N05 TransformerClassifier state dicts."""
    def __init__(self, d_model, max_len=64):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)[:, :pe[:, 1::2].shape[1]]
        self.register_buffer("pe", pe.unsqueeze(0))  # key: "pe"
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerClassifier(nn.Module):
    """Matches the TransformerClassifier saved in N05 models/:
       self.input_proj, self.pos_encoder, self.encoder, self.dropout, self.classifier."""
    def __init__(self, input_size=N_FEATS, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128,
                 num_classes=N_CLASSES, dropout=0.2):
        super().__init__()
        self.input_proj  = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=WINDOW + 1)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu")
        self.encoder     = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.dropout     = nn.Dropout(dropout)
        self.classifier  = nn.Linear(d_model, num_classes)
    def forward(self, x):
        x = self.pos_encoder(self.input_proj(x))
        x = self.encoder(x).mean(dim=1)
        return self.classifier(self.dropout(x))


def inverse_freq_weights(y, cap=10.0):
    counts = np.bincount(y, minlength=N_CLASSES).astype(np.float64)
    N = counts.sum()
    w = np.zeros(N_CLASSES, dtype=np.float64)
    present = counts > 0
    w[present] = N / (present.sum() * counts[present])
    w[present] /= w[present].mean()
    w = np.minimum(w, cap)
    if (w > 0).any(): w[w > 0] /= w[w > 0].mean()
    return w.astype(np.float32)

print("Model classes defined — layer names match saved state dicts.")

Model classes defined — layer names match saved state dicts.


## Section 5 — Training-set evaluation

This section evaluates the saved local and centralized models on their training data. These values are then compared with the validation metrics recorded at the best epoch. A large train-validation gap would suggest overfitting, while small gaps indicate acceptable generalization behaviour.


In [44]:
# 3.1 — Evaluate local models (one per client per architecture)
def load_and_eval_local(model_class, model_kwargs, models_dir, label_tmpl, client):
    """Load state dict and evaluate on client training data."""
    path = models_dir / (label_tmpl.format(c=client.lower()) + ".pt")
    if not path.exists():
        print(f"  Missing: {path}"); return None
    model = model_class(**model_kwargs).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    # class weights from this client's training data
    weights = inverse_freq_weights(data[client]["y"])
    loss, acc, f1 = eval_model(model, data[client]["X"], data[client]["y"], weights)
    return {"loss": loss, "accuracy": acc, "macro_f1": f1}

GRU_KWARGS  = {"input_size": N_FEATS, "hidden_size": 64, "num_layers": 1,
               "num_classes": N_CLASSES, "dropout": 0.2}
LSTM_KWARGS = GRU_KWARGS.copy()
TX_KWARGS   = {"input_size": N_FEATS, "d_model": 64, "nhead": 4,
               "num_layers": 2, "dim_feedforward": 128,
               "num_classes": N_CLASSES, "dropout": 0.2}

local_configs = [
    ("Local GRU",         GRUClassifier,         GRU_KWARGS,  GRU_MODELS,  "local_gru_{c}"),
    ("Local LSTM",        LSTMClassifier,        LSTM_KWARGS, LSTM_MODELS, "local_lstm_{c}"),
    ("Local Transformer", TransformerClassifier, TX_KWARGS,   TX_MODELS,   "local_transformer_{c}"),
]

local_train_results = {}
for label, cls, kwargs, mdir, tmpl in local_configs:
    print(f"\nEvaluating {label} on training sets...")
    local_train_results[label] = {}
    for client in ACTIVE_CLIENTS:
        r = load_and_eval_local(cls, kwargs, mdir, tmpl, client)
        if r: local_train_results[label][client] = r
        print(f"  {client}: train_loss={r['loss']:.4f} | train_acc={r['accuracy']:.4f}")

print("\nLocal training evaluation done.")


Evaluating Local GRU on training sets...
  England: train_loss=0.6186 | train_acc=0.7244
  Ireland: train_loss=0.7379 | train_acc=0.8277
  USA: train_loss=0.9605 | train_acc=0.8303

Evaluating Local LSTM on training sets...
  England: train_loss=0.6082 | train_acc=0.7346
  Ireland: train_loss=0.7291 | train_acc=0.8296
  USA: train_loss=0.8883 | train_acc=0.8553

Evaluating Local Transformer on training sets...
  England: train_loss=0.5779 | train_acc=0.7426
  Ireland: train_loss=0.6436 | train_acc=0.8457
  USA: train_loss=0.8868 | train_acc=0.8906

Local training evaluation done.


In [45]:
# 3.2 — Evaluate centralized models on pooled training data
X_pool = np.concatenate([data[c]["X"] for c in ACTIVE_CLIENTS])
y_pool = np.concatenate([data[c]["y"] for c in ACTIVE_CLIENTS])
pool_weights = inverse_freq_weights(y_pool)

central_configs = [
    ("Centralized GRU",         GRUClassifier,         GRU_KWARGS,  GRU_MODELS,  "centralized_gru"),
    ("Centralized LSTM",        LSTMClassifier,        LSTM_KWARGS, LSTM_MODELS, "centralized_lstm"),
    ("Centralized Transformer", TransformerClassifier, TX_KWARGS,   TX_MODELS,   "centralized_transformer"),
]

central_train_results = {}
for label, cls, kwargs, mdir, fname in central_configs:
    path = mdir / f"{fname}.pt"
    if not path.exists():
        print(f"Missing: {path}"); continue
    model = cls(**kwargs).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    loss, acc, f1 = eval_model(model, X_pool, y_pool, pool_weights)
    central_train_results[label] = {"loss": loss, "accuracy": acc, "macro_f1": f1}
    print(f"{label}: train_loss={loss:.4f} | train_acc={acc:.4f}")

print("\nCentralized training evaluation done.")

Centralized GRU: train_loss=0.5101 | train_acc=0.8069
Centralized LSTM: train_loss=0.5087 | train_acc=0.8038
Centralized Transformer: train_loss=0.5118 | train_acc=0.8105

Centralized training evaluation done.


## Section 6 — Load validation history

The validation loss, validation accuracy, and validation macro-F1 are read from the training-history CSV files saved by the baseline notebooks. The best epoch is selected using validation macro-F1, which is the same criterion used during training.


In [46]:
# 4 — Load best-epoch val metrics from training history CSVs
# Print actual audit directory contents so we can verify filenames
for arch, adir in [("GRU", GRU_DIR), ("LSTM", LSTM_DIR), ("Transformer", TX_DIR)]:
    audit = adir.parent / "audit"
    hist_files = sorted(audit.glob("*history*.csv"))
    print(f"\n{arch} audit history files:")
    for f in hist_files: print(f"  {f.name}")

def read_best_val(history_path):
    if not history_path.exists():
        print(f"  [SKIP] not found: {history_path.name}")
        return None
    df = pd.read_csv(history_path)
    best = df.loc[df["val_macro_f1"].idxmax()]
    return {
        "best_epoch":   int(best["epoch"]),
        "val_loss":     round(float(best["val_loss"]), 4),
        "val_accuracy": round(float(best["val_accuracy"]), 4),
        "val_macro_f1": round(float(best["val_macro_f1"]), 4),
    }

val_metrics = {}

# N03 GRU — files saved WITHOUT "gru_" prefix
# local_history_{client}.csv, centralized_training_history.csv
gru_audit = GRU_DIR.parent / "audit"
for client in ACTIVE_CLIENTS:
    c = client.lower()
    r = read_best_val(gru_audit / f"local_history_{c}.csv")
    if r: val_metrics[f"Local GRU | {client}"] = r

r = read_best_val(gru_audit / "centralized_training_history.csv")
if r: val_metrics["Centralized GRU"] = r

# N04 LSTM — files saved WITH "lstm_" prefix
# local_lstm_history_{client}.csv, centralized_lstm_training_history.csv
lstm_audit = LSTM_DIR.parent / "audit"
for client in ACTIVE_CLIENTS:
    c = client.lower()
    r = read_best_val(lstm_audit / f"local_lstm_history_{c}.csv")
    if r: val_metrics[f"Local LSTM | {client}"] = r

r = read_best_val(lstm_audit / "centralized_lstm_training_history.csv")
if r: val_metrics["Centralized LSTM"] = r

# N05 Transformer — files saved WITH "transformer_" prefix
# local_transformer_history_{client}.csv, centralized_transformer_training_history.csv
tx_audit = TX_DIR.parent / "audit"
for client in ACTIVE_CLIENTS:
    c = client.lower()
    r = read_best_val(tx_audit / f"local_transformer_history_{c}.csv")
    if r: val_metrics[f"Local Transformer | {client}"] = r

r = read_best_val(tx_audit / "centralized_transformer_training_history.csv")
if r: val_metrics["Centralized Transformer"] = r

print(f"\nLoaded val metrics for {len(val_metrics)} model/client combinations:")
for k in val_metrics: print(f"  {k}")


GRU audit history files:
  centralized_training_history.csv
  history_fedavg_v1.csv
  history_fedavg_v2.csv
  history_fedavg_v3.csv
  history_fedprox_v1.csv
  history_fedprox_v2.csv
  history_fedprox_v3.csv
  local_history_england.csv
  local_history_ireland.csv
  local_history_usa.csv

LSTM audit history files:
  centralized_lstm_training_history.csv
  local_lstm_history_england.csv
  local_lstm_history_ireland.csv
  local_lstm_history_usa.csv
  lstm_history_fedavg_v1.csv
  lstm_history_fedavg_v2.csv
  lstm_history_fedavg_v3.csv
  lstm_history_fedprox_v1.csv
  lstm_history_fedprox_v2.csv
  lstm_history_fedprox_v3.csv

Transformer audit history files:
  alternative_transformer_training_history.csv
  centralized_transformer_training_history.csv
  local_transformer_history_england.csv
  local_transformer_history_ireland.csv
  local_transformer_history_usa.csv
  transformer_history_fedavg_v1.csv
  transformer_history_fedavg_v2.csv
  transformer_history_fedavg_v3.csv
  transformer_history

## Section 7 — Train-vs-validation comparison

The table below compares each model at its best validation epoch. The most useful diagnostic is the accuracy gap, computed as validation accuracy minus training accuracy. Small positive or slightly negative gaps are acceptable. A strong negative gap would be a warning sign of overfitting.

**Note.** The previous training notebooks saved train loss and validation accuracy per epoch, but not train accuracy per epoch. For this reason, this notebook computes train accuracy once from the saved best checkpoint and compares it with the validation accuracy of the best epoch. This is enough for the overfitting check because the saved checkpoint is exactly the model used later for testing.


In [47]:

# 5 — Build the train vs val comparison table
# Accuracy is kept for readability, but Macro-F1 is added because it is the thesis primary metric.
rows = []

for label, cls, kwargs, mdir, tmpl in local_configs:
    for client in ACTIVE_CLIENTS:
        key_val = f"{label} | {client}"
        if client not in local_train_results.get(label, {}):
            continue
        tr = local_train_results[label][client]
        vl = val_metrics.get(key_val, {})
        rows.append({
            "Model":          label,
            "Client":         client,
            "Best_epoch":     vl.get("best_epoch", "—"),
            "Train_loss":     round(tr["loss"], 4),
            "Val_loss":       vl.get("val_loss", "—"),
            "Δ_loss":         round(vl.get("val_loss", 0) - tr["loss"], 4)
                              if vl.get("val_loss") else "—",
            "Train_acc":      round(tr["accuracy"], 4),
            "Val_acc":        vl.get("val_accuracy", "—"),
            "Δ_acc":          round(vl.get("val_accuracy", 0) - tr["accuracy"], 4)
                              if vl.get("val_accuracy") else "—",
            "Train_macro_f1": round(tr["macro_f1"], 4),
            "Val_macro_f1":   vl.get("val_macro_f1", "—"),
            "Δ_macro_f1":     round(vl.get("val_macro_f1", 0) - tr["macro_f1"], 4)
                              if vl.get("val_macro_f1") else "—",
        })

for label, cls, kwargs, mdir, fname in central_configs:
    if label not in central_train_results:
        continue
    tr = central_train_results[label]
    vl = val_metrics.get(label, {})
    rows.append({
        "Model":          label,
        "Client":         "Pooled",
        "Best_epoch":     vl.get("best_epoch", "—"),
        "Train_loss":     round(tr["loss"], 4),
        "Val_loss":       vl.get("val_loss", "—"),
        "Δ_loss":         round(vl.get("val_loss", 0) - tr["loss"], 4)
                          if vl.get("val_loss") else "—",
        "Train_acc":      round(tr["accuracy"], 4),
        "Val_acc":        vl.get("val_accuracy", "—"),
        "Δ_acc":          round(vl.get("val_accuracy", 0) - tr["accuracy"], 4)
                          if vl.get("val_accuracy") else "—",
        "Train_macro_f1": round(tr["macro_f1"], 4),
        "Val_macro_f1":   vl.get("val_macro_f1", "—"),
        "Δ_macro_f1":     round(vl.get("val_macro_f1", 0) - tr["macro_f1"], 4)
                          if vl.get("val_macro_f1") else "—",
    })

overfitting_df = pd.DataFrame(rows)
overfitting_df.to_csv(OUTPUT_DIR / "train_vs_val_comparison.csv", index=False)
print("Train vs Val comparison with Macro-F1 gap:")
display(overfitting_df)


Train vs Val comparison with Macro-F1 gap:


,Model,Client,Best_epoch,Train_loss,Val_loss,Δ_loss,Train_acc,Val_acc,Δ_acc,Train_macro_f1,Val_macro_f1,Δ_macro_f1
0,Local GRU,England,21,0.6186,0.5572,-0.0614,0.7244,0.7682,0.0438,0.7093,0.7281,0.0188
1,Local GRU,Ireland,20,0.7379,0.7057,-0.0322,0.8277,0.8140,-0.0137,0.4884,0.3083,-0.1801
2,Local GRU,USA,3,0.9605,0.5390,-0.4215,0.8303,0.9441,0.1138,0.2933,0.3340,0.0407
3,Local LSTM,England,25,0.6082,0.5507,-0.0575,0.7346,0.7755,0.0409,0.7222,0.7403,0.0181
4,Local LSTM,Ireland,17,0.7291,0.7190,-0.0101,0.8296,0.8308,0.0012,0.4952,0.3188,-0.1764
5,Local LSTM,USA,9,0.8883,0.4770,-0.4113,0.8553,0.9463,0.0910,0.3081,0.3361,0.0280
6,Local Transformer,England,37,0.5779,0.5087,-0.0692,0.7426,0.7841,0.0415,0.7287,0.7411,0.0124
7,Local Transformer,Ireland,20,0.6436,0.7419,0.0983,0.8457,0.8433,-0.0024,0.5383,0.3275,-0.2108
8,Local Transformer,USA,9,0.8868,0.5241,-0.3627,0.8906,0.9593,0.0687,0.3492,0.3464,-0.0028
9,Centralized GRU,Pooled,29,0.5101,0.4944,-0.0157,0.8069,0.8267,0.0198,0.7249,0.7373,0.0124


## Section 8 — Accuracy-gap plot

The bar chart visualizes the train-validation accuracy gap for the twelve non-federated deep-learning models. The dashed band around zero is used as a practical reading aid: values close to zero indicate that validation performance is close to training performance.


In [48]:
# 6 — Plot: Δ accuracy (val_acc − train_acc) per model — small positive = no overfitting
fig, ax = plt.subplots(figsize=(12, 4))
labels_plot = [f"{r['Model']}\n{r['Client']}" for _, r in overfitting_df.iterrows()
               if isinstance(r["Δ_acc"], (int, float))]
deltas = [r["Δ_acc"] for _, r in overfitting_df.iterrows()
          if isinstance(r["Δ_acc"], (int, float))]
bar_colors = ["#2E8B57" if d >= 0 else "#C0392B" for d in deltas]
ax.bar(range(len(deltas)), deltas, color=bar_colors, width=0.6)
ax.axhline(0, color="black", lw=0.8)
ax.axhline(0.05, color="gray", ls="--", lw=0.8, alpha=0.6, label="+0.05 threshold")
ax.axhline(-0.05, color="gray", ls="--", lw=0.8, alpha=0.6)
ax.set_xticks(range(len(labels_plot)))
ax.set_xticklabels(labels_plot, fontsize=6.5, rotation=25, ha="right")
ax.set_ylabel("Δ accuracy (val − train)")
ax.set_title("Overfitting check: val_accuracy − train_accuracy at best epoch\n"
             "Values near 0 or slightly negative = good generalisation")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overfitting_delta_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 8b — Macro-F1 gap plot

This plot aligns the overfitting diagnostic with the thesis primary metric, present-class Macro-F1.


In [49]:
# 6b — Plot: Δ Macro-F1 (val_macro_f1 − train_macro_f1) per model
fig, ax = plt.subplots(figsize=(12, 4))

valid_rows = overfitting_df[pd.to_numeric(overfitting_df["Δ_macro_f1"], errors="coerce").notna()].copy()
labels_plot = [f"{row['Model']}\n{row['Client']}" for _, row in valid_rows.iterrows()]
deltas = valid_rows["Δ_macro_f1"].astype(float).tolist()

bar_colors = ["#2E8B57" if d >= -0.05 else "#C0392B" for d in deltas]
ax.bar(range(len(deltas)), deltas, color=bar_colors, width=0.6)
ax.axhline(0, color="black", lw=0.8)
ax.axhline(0.05, color="gray", ls="--", lw=0.8, alpha=0.6, label="±0.05 reference")
ax.axhline(-0.05, color="gray", ls="--", lw=0.8, alpha=0.6)
ax.set_xticks(range(len(labels_plot)))
ax.set_xticklabels(labels_plot, fontsize=6.5, rotation=25, ha="right")
ax.set_ylabel("Δ Macro-F1 (val − train)")
ax.set_title("Overfitting check using Macro-F1 gap at the selected validation epoch")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overfitting_delta_macro_f1.png", dpi=150, bbox_inches="tight")
plt.show()


## Section 9 — Training curves

The following figures show training loss, validation loss, and validation accuracy across epochs for each architecture. The expected healthy pattern is that training and validation curves move together without a strong divergence.


In [50]:
# 7 — Per-architecture training curves from history CSVs
# Plot train_loss + val_loss per epoch for each best model
arch_info = [
    ("GRU",   GRU_DIR,  "gru"),
    ("LSTM",  LSTM_DIR, "lstm"),
    ("Transformer", TX_DIR, "transformer"),
]

for arch, adir, prefix in arch_info:
    audit = adir.parent / "audit"
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
    # GRU files have no architecture prefix; LSTM/Transformer do
    p = "" if prefix == "gru" else f"{prefix}_"
    for ax, (label, hist_file) in zip(axes, [
        ("England (local)",  f"local_{p}history_england.csv"),
        ("Ireland (local)",  f"local_{p}history_ireland.csv"),
        ("USA (local)",      f"local_{p}history_usa.csv"),
        ("Pooled (central)", f"centralized_{p}training_history.csv"),
    ]):
        path = audit / hist_file
        if not path.exists():
            ax.set_visible(False); continue
        df = pd.read_csv(path)
        ax2 = ax.twinx()
        ax.plot(df["epoch"], df["train_loss"], "o-", color="#C0392B", ms=2.5, label="train loss")
        ax.plot(df["epoch"], df["val_loss"],   "s-", color="#E67E22", ms=2.5, label="val loss")
        ax2.plot(df["epoch"], df["val_accuracy"], "^-", color="#1F3A5F", ms=2.5, label="val acc")
        ax.set_title(label, fontsize=8)
        ax.set_xlabel("Epoch", fontsize=7)
        ax.set_ylabel("Loss", fontsize=7, color="#7B0000")
        ax2.set_ylabel("Val acc", fontsize=7, color="#1F3A5F")
        ax.grid(True, alpha=0.3)
        l1,la1 = ax.get_legend_handles_labels()
        l2,la2 = ax2.get_legend_handles_labels()
        ax.legend(l1+l2, la1+la2, fontsize=6, loc="upper right")
    fig.suptitle(f"{arch} — training curves (train loss / val loss / val accuracy per epoch)",
                 fontsize=9, fontweight="bold")
    plt.tight_layout()
    out_path = OUTPUT_DIR / f"training_curves_{prefix}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

Saved: /kaggle/working/overfitting_evidence/training_curves_gru.png
Saved: /kaggle/working/overfitting_evidence/training_curves_lstm.png
Saved: /kaggle/working/overfitting_evidence/training_curves_transformer.png


## Section 10 — Save outputs

The generated figures and tables are saved in one output folder and zipped for download. These outputs can be used in Chapter 4 to support the discussion of generalization and overfitting.


In [51]:
# 8 — Zip for download
import shutil
zip_path = shutil.make_archive(
    base_name=str(Path("/kaggle/working") / "overfitting_evidence"),
    format="zip", root_dir=OUTPUT_DIR)
print(f"Archive: {zip_path}")
print("\nFiles produced:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}")

Archive: /kaggle/working/overfitting_evidence.zip

Files produced:
  overfitting_delta_accuracy.png
  overfitting_delta_macro_f1.png
  train_vs_val_comparison.csv
  training_curves_gru.png
  training_curves_lstm.png
  training_curves_transformer.png


## Section 11 — Reading for the thesis

The train-vs-validation comparison does not show a problematic overfitting pattern. Most validation accuracies are close to, or slightly higher than, the corresponding training accuracies. This can happen because dropout is active during training and disabled during validation, and because validation sets for Ireland and USA are strongly dominated by the Excellent class.

The key interpretation is that the main limitation of the models is not classical overfitting. Instead, the results remain mainly limited by class imbalance, client heterogeneity, and repeated/modal-value patterns already documented in the previous notebooks. Therefore, this notebook supports the reliability of the reported baselines while keeping the data-related limitations visible.
